# 03. Feature Engineering

This notebook converts the findings from exploratory data analysis into reproducible dataset versions for model training and comparison.

In [ ]:
from pathlib import Path
import json
import sys

## Environment Setup

In [20]:
PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print("Project root configured.")

PROCESSED_DIR = PROJECT_ROOT / "data/processed"
print("Processed directory configured.")

Project root configured.
Processed directory configured.


In [ ]:
from src.churn_ml.data import load_competition_data
from src.churn_ml.features import PreparedDataset
from src.churn_ml.features import (
    save_dataset,
)

## Metadata Loading

In [22]:
metadata_dir = PROJECT_ROOT / "data/interim/eda_metadata"


def load_feature_metadata(filename: str) -> list[str]:
    with (metadata_dir / filename).open("r", encoding="utf-8") as file:
        return json.load(file)


numeric_features = load_feature_metadata("numeric_features.json")
categorical_features = load_feature_metadata("categorical_features.json")
constant_features = load_feature_metadata("constant_features.json")
all_missing_features = load_feature_metadata("all_missing_features.json")
very_high_missing_features = load_feature_metadata("very_high_missing_features.json")

print(f"Numeric features:              {len(numeric_features)}")
print(f"Categorical features:          {len(categorical_features)}")
print(f"Constant features:             {len(constant_features)}")
print(f"All-missing features:          {len(all_missing_features)}")
print(f"Very high-missing features:    {len(very_high_missing_features)}")

Numeric features:              192
Categorical features:          38
Constant features:             25
All-missing features:          18
Very high-missing features:    136


## Dataset Loading

In [23]:
data = load_competition_data(
    train_path=PROJECT_ROOT / "data/raw/final_proj_data.csv",
    test_path=PROJECT_ROOT / "data/raw/final_proj_test.csv",
    sample_submission_path=(PROJECT_ROOT / "data/raw/final_proj_sample_submission.csv"),
)

print(f"Train features: {data.X.shape}")
print(f"Target:         {data.y.shape}")
print(f"Test features:  {data.test.shape}")

Train features: (10000, 230)
Target:         (10000,)
Test features:  (2500, 230)


## Data & Metadata Validation

In [24]:
train_columns = set(data.X.columns)

metadata_sets = {
    "numeric_features": set(numeric_features),
    "categorical_features": set(categorical_features),
    "constant_features": set(constant_features),
    "all_missing_features": set(all_missing_features),
    "very_high_missing_features": set(very_high_missing_features),
}

for name, features in metadata_sets.items():
    missing = features - train_columns
    extra = (
        train_columns - features
        if name
        in {
            "numeric_features",
            "categorical_features",
        }
        else set()
    )

    print(f"\n{name}")

    if missing:
        print(f"  Missing in dataset: {len(missing)}")
    else:
        print("  ✓ All metadata features exist in dataset")

    if extra:
        print(f"  Unclassified features: {len(extra)}")


numeric_features
  ✓ All metadata features exist in dataset
  Unclassified features: 38

categorical_features
  ✓ All metadata features exist in dataset
  Unclassified features: 192

constant_features
  ✓ All metadata features exist in dataset

all_missing_features
  ✓ All metadata features exist in dataset

very_high_missing_features
  ✓ All metadata features exist in dataset


## Dataset Version v0: Raw Minimal

In [25]:
features_to_drop = sorted(set(constant_features) | set(all_missing_features))

X_train_v0 = data.X.drop(columns=features_to_drop).copy()
X_test_v0 = data.test.drop(columns=features_to_drop).copy()

dataset_v0 = PreparedDataset(
    version="v0_raw_minimal",
    X_train=X_train_v0,
    y_train=data.y.copy(),
    X_test=X_test_v0,
    metadata={
        "description": (
            "Raw feature set with constant and all-missing features removed."
        ),
        "source": "raw",
        "dropped_features": features_to_drop,
        "n_dropped_features": len(features_to_drop),
        "n_train_rows": len(X_train_v0),
        "n_test_rows": len(X_test_v0),
        "n_features": X_train_v0.shape[1],
    },
)

print(f"Dataset version:   {dataset_v0.version}")
print(f"Train shape:       {dataset_v0.X_train.shape}")
print(f"Target shape:      {dataset_v0.y_train.shape}")
print(f"Test shape:        {dataset_v0.X_test.shape}")
print(f"Dropped features:  {dataset_v0.metadata['n_dropped_features']}")

Dataset version:   v0_raw_minimal
Train shape:       (10000, 205)
Target shape:      (10000,)
Test shape:        (2500, 205)
Dropped features:  25


In [26]:
saved_path = save_dataset(
    dataset=dataset_v0, processed_dir=PROCESSED_DIR, overwrite=True
)

print(f"Saved to: {saved_path}")

Saved to: c:\Users\pbori\Documents\Courses\Neoversity\ML Foundations\neoversity-ml-final\data\processed\v0_raw_minimal
